In [1]:
# Dataset Link
# https://archive.ics.uci.edu/dataset/53/iris

# https://scikit-learn.org/dev/modules/generated/sklearn.datasets.load_iris.html
#sklearn.datasets.load_iris

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder



In [3]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns = iris.feature_names)
df['target'] = iris.target


In [4]:
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [6]:
train_df , test_df = train_test_split(df, test_size = 0.2, random_state = 42, stratify = df["target"] )

In [9]:
X_train, Y_train = train_df.drop(columns=['target']), train_df['target']
X_test, Y_test = test_df.drop(columns=['target']), test_df['target']



In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [11]:
X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype = torch.long)
Y_test_tensor = torch.tensor(Y_test.values, dtype = torch.long)

In [19]:
class IrisClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(IrisClassifier, self).__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim)
    )

  def forward(self, x):
      return self.network(x)

In [20]:
X_train.shape

(120, 4)

In [21]:
input_dim = X_train.shape[1]
hidden_dim = 16
output_dim = 3

In [22]:
model = IrisClassifier(input_dim, hidden_dim, output_dim)

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

In [25]:
# Train the model

epochs = 500

for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()

  predictions = model(X_train_tensor)
  loss = criterion(predictions, Y_train_tensor)
  loss.backward()
  optimizer.step()


  if (epoch + 1) % 50 == 0:
    print(f"Epoch [{epoch+1}/{epochs}, loss : {loss.item(): .4f}]")

Epoch [50/500, loss :  0.0813]
Epoch [100/500, loss :  0.0374]
Epoch [150/500, loss :  0.0333]
Epoch [200/500, loss :  0.0270]
Epoch [250/500, loss :  0.0172]
Epoch [300/500, loss :  0.0082]
Epoch [350/500, loss :  0.0017]
Epoch [400/500, loss :  0.0007]
Epoch [450/500, loss :  0.0004]
Epoch [500/500, loss :  0.0003]


In [26]:
model.eval()
with torch.no_grad():
  Y_pred = model(X_test_tensor)
  Y_pred_labels = torch.argmax(Y_pred, dim = 1)
  accuracy = (Y_pred_labels == Y_test_tensor).sum().item()/ Y_test_tensor.size(0)
  print(f"Test accuracy : {accuracy:.4f}")

Test accuracy : 0.9667


In [31]:
def predict_iris(sepal_length, sepal_width, petal_length, petal_width):
  input_data =  np.array([[sepal_length, sepal_width, petal_length, petal_width]])
  input_data = scaler.transform(input_data)
  input_tensor = torch.tensor(input_data, dtype = torch.float32)
  model.eval()
  with torch.no_grad():
    prediction = model(input_tensor)
    prediction_label = torch.argmax(prediction, dim = 1).item()
  return iris.target_names[prediction_label]

In [32]:
predicted_class = predict_iris(5.1, 3.5, 1.4, 0.2)
print(f"Predicted species : {predicted_class}")

Predicted species : setosa


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
